# 確率の公理と性質 — インタラクティブデモ

このノートブックでは、**コルモゴロフの確率の公理**とそこから導かれる重要な性質を、
シミュレーションと可視化を通じて体験的に学びます。

---
## 目次
1. ライブラリのインポート
2. 確率の公理（コルモゴロフ）
3. 公理1：非負性のデモ
4. 公理2：全確率 = 1 のデモ
5. 公理3：加法性のデモ（排反事象）
6. 導出される性質のデモ
   - 補事象の確率
   - 包除原理
   - 単調性
7. 大数の法則との接続（おまけ）

## 1. ライブラリのインポート

In [ ]:
# ── 必要なパッケージのインストール ──────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q', 'matplotlib-venn', 'japanize-matplotlib'], check=True)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import japanize_matplotlib          # ← これだけで日本語フォントが有効になる
from matplotlib_venn import venn2, venn3
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120
print('ライブラリの読み込み完了 ✓')

---
## 2. 確率の公理（コルモゴロフの公理）

標本空間 $\Omega$ 上の確率測度 $P$ は以下の3つの公理を満たす：

| 公理 | 内容 | 数式 |
|------|------|------|
| **公理1** | 非負性 | $P(A) \geq 0 \quad \forall A \subseteq \Omega$ |
| **公理2** | 全確率 | $P(\Omega) = 1$ |
| **公理3** | 加法性 | $A \cap B = \emptyset \Rightarrow P(A \cup B) = P(A) + P(B)$ |

> これら3公理のみから、確率論の全ての性質が導出できます。

---
## 3. 公理1：非負性のデモ

サイコロ投げを例に、各事象の相対頻度（経験確率）が常に $\geq 0$ であることを確認します。

In [ ]:
# サイコロを n 回振るシミュレーション
n = 10000
rolls = np.random.randint(1, 7, size=n)

faces = np.arange(1, 7)
freq = np.array([(rolls == f).sum() / n for f in faces])

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(faces, freq, color='steelblue', edgecolor='white', linewidth=1.2)
ax.axhline(1/6, color='crimson', linestyle='--', linewidth=1.5, label='理論値 1/6 ≈ 0.1667')
ax.set_ylim(0, 0.25)
ax.set_xlabel('出た目', fontsize=12)
ax.set_ylabel('相対頻度（経験確率）', fontsize=12)
ax.set_title(f'公理1：非負性　— サイコロ {n:,} 回のシミュレーション', fontsize=13)
ax.legend(fontsize=11)

for bar, f in zip(bars, freq):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{f:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print('各事象の経験確率（すべて ≥ 0）:')
for f, p in zip(faces, freq):
    print(f'  P(目={f}) = {p:.4f}  >= 0 : {p >= 0}')

---
## 4. 公理2：全確率 = 1 のデモ

標本空間全体の確率 $P(\Omega)$ を累積して 1 に収束していく様子を観察します。

In [ ]:
# 全事象の確率の累積和
# サイコロ：P(1)+P(2)+...+P(6) が 1 になることを段階的に示す
n_sim = 5000
rolls = np.random.randint(1, 7, size=n_sim)

ns = np.arange(100, n_sim+1, 100)
totals = []
for k in ns:
    subset = rolls[:k]
    total = sum((subset == f).mean() for f in range(1, 7))
    totals.append(total)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左：全確率の収束
axes[0].plot(ns, totals, color='steelblue', linewidth=1.5)
axes[0].axhline(1.0, color='crimson', linestyle='--', linewidth=1.5, label='P(Ω) = 1')
axes[0].set_ylim(0.95, 1.05)
axes[0].set_xlabel('試行回数', fontsize=12)
axes[0].set_ylabel('Σ P(目=k) の累積', fontsize=12)
axes[0].set_title('公理2：全確率 P(Ω) = 1 への収束', fontsize=12)
axes[0].legend(fontsize=11)

# 右：円グラフで各事象の分割
freq_final = [(rolls == f).mean() for f in range(1, 7)]
labels = [f'目={f}\n({p:.3f})' for f, p in zip(range(1, 7), freq_final)]
colors = plt.cm.Set2(np.linspace(0, 1, 6))
axes[1].pie(freq_final, labels=labels, colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title(f'標本空間の分割（合計 = {sum(freq_final):.4f}）', fontsize=12)

plt.tight_layout()
plt.show()

print(f'全確率の合計 = {sum(freq_final):.6f}  (理論値 = 1.000000)')

---
## 5. 公理3：加法性のデモ（排反事象）

$A \cap B = \emptyset$ のとき、$P(A \cup B) = P(A) + P(B)$

サイコロの例：$A = \{1, 2\}$、$B = \{5, 6\}$ は排反事象。

In [ ]:
n = 20000
rolls = np.random.randint(1, 7, size=n)

A = {1, 2}   # 目が1か2
B = {5, 6}   # 目が5か6

pA = np.isin(rolls, list(A)).mean()
pB = np.isin(rolls, list(B)).mean()
pAuB = np.isin(rolls, list(A | B)).mean()
pAuB_theory = pA + pB  # 排反なので加法性が成立

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左：棒グラフ比較
labels = ['P(A)', 'P(B)', 'P(A∪B)\n（実測）', 'P(A)+P(B)\n（加法性）']
values = [pA, pB, pAuB, pAuB_theory]
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
bars = axes[0].bar(labels, values, color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_ylim(0, 0.8)
axes[0].set_ylabel('確率', fontsize=12)
axes[0].set_title('公理3：加法性の確認\nA={1,2}, B={5,6}（排反）', fontsize=12)
for bar, v in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=10)

# 右：ベン図（排反 = 重ならない）
v = venn2(subsets=(pA, pB, 0), set_labels=('A = {1,2}', 'B = {5,6}'),
          ax=axes[1], set_colors=('#4C72B0', '#DD8452'), alpha=0.6)
axes[1].set_title('排反事象のベン図\n（A∩B = ∅）', fontsize=12)

plt.tight_layout()
plt.show()

print(f'P(A)           = {pA:.4f}  （理論値 2/6 = {2/6:.4f}）')
print(f'P(B)           = {pB:.4f}  （理論値 2/6 = {2/6:.4f}）')
print(f'P(A∪B) 実測値  = {pAuB:.4f}  （理論値 4/6 = {4/6:.4f}）')
print(f'P(A)+P(B)      = {pAuB_theory:.4f}')
print(f'差（|実測 - 加法性|） = {abs(pAuB - pAuB_theory):.6f}')

---
## 6. 導出される性質のデモ

3公理から**定理として**導かれる重要な性質を確認します。

### 6-1. 補事象の確率：$P(A^c) = 1 - P(A)$

In [ ]:
n = 10000
rolls = np.random.randint(1, 7, size=n)

# 複数の事象で補事象の関係を確認
events = [
    ('偶数（{2,4,6}）', lambda x: x % 2 == 0),
    ('3以下（{1,2,3}）', lambda x: x <= 3),
    ('6の目（{6}）',    lambda x: x == 6),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, (name, cond) in zip(axes, events):
    pA = cond(rolls).mean()
    pAc = (~cond(rolls)).mean()
    total = pA + pAc

    ax.bar(['P(A)', 'P(Aᶜ)', 'P(A)+P(Aᶜ)'],
           [pA, pAc, total],
           color=['#4C72B0', '#DD8452', '#55A868'],
           edgecolor='white')
    ax.axhline(1.0, color='crimson', linestyle='--', linewidth=1.2)
    ax.set_ylim(0, 1.2)
    ax.set_title(f'A = {name}\nP(A)={pA:.3f}, P(Aᶜ)={pAc:.3f}', fontsize=10)
    ax.set_ylabel('確率', fontsize=10)
    for i, v in enumerate([pA, pAc, total]):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)

fig.suptitle('補事象の確率：P(Aᶜ) = 1 − P(A)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 6-2. 包除原理：$P(A \cup B) = P(A) + P(B) - P(A \cap B)$

一般の（排反でない）事象に対する加法の公式です。

In [ ]:
n = 30000
rolls = np.random.randint(1, 7, size=n)

# A = 偶数 {2,4,6}、B = 4以上 {4,5,6}  → 重なり = {4,6}
A_mask = rolls % 2 == 0
B_mask = rolls >= 4

pA   = A_mask.mean()
pB   = B_mask.mean()
pAB  = (A_mask & B_mask).mean()   # P(A∩B)
pAuB = (A_mask | B_mask).mean()   # P(A∪B) 実測
pAuB_ie = pA + pB - pAB           # 包除原理

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左：数値比較
cats = ['P(A)', 'P(B)', 'P(A∩B)', 'P(A∪B)\n実測', 'P(A)+P(B)\n−P(A∩B)']
vals = [pA, pB, pAB, pAuB, pAuB_ie]
colors = ['#4C72B0','#DD8452','#C44E52','#55A868','#8172B2']
bars = axes[0].bar(cats, vals, color=colors, edgecolor='white')
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel('確率', fontsize=11)
axes[0].set_title('包除原理の確認\nA=偶数, B=4以上', fontsize=12)
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.01,
                f'{v:.4f}', ha='center', fontsize=8.5)

# 右：ベン図
v = venn2(subsets=(pA - pAB, pB - pAB, pAB),
          set_labels=('A = 偶数\n{2,4,6}', 'B = 4以上\n{4,5,6}'),
          ax=axes[1], set_colors=('#4C72B0','#DD8452'), alpha=0.6)
axes[1].set_title('ベン図（A∩B = {4,6}）', fontsize=12)

plt.tight_layout()
plt.show()

print(f'P(A)              = {pA:.4f}  （理論値 {3/6:.4f}）')
print(f'P(B)              = {pB:.4f}  （理論値 {3/6:.4f}）')
print(f'P(A∩B)            = {pAB:.4f}  （理論値 {2/6:.4f}）')
print(f'P(A∪B) 実測       = {pAuB:.4f}  （理論値 {4/6:.4f}）')
print(f'P(A)+P(B)−P(A∩B) = {pAuB_ie:.4f}')
print(f'差                 = {abs(pAuB - pAuB_ie):.6f}')

### 6-3. 単調性：$A \subseteq B \Rightarrow P(A) \leq P(B)$

In [ ]:
n = 20000
rolls = np.random.randint(1, 7, size=n)

# 入れ子になった事象の列 A1 ⊂ A2 ⊂ A3 ⊂ A4
nested = [
    ('{1}',       rolls == 1),
    ('{1,2}',     rolls <= 2),
    ('{1,2,3}',   rolls <= 3),
    ('{1,2,3,4}', rolls <= 4),
]

labels = [name for name, _ in nested]
probs  = [mask.mean() for _, mask in nested]
theory = [1/6, 2/6, 3/6, 4/6]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, probs,   width, label='シミュレーション', color='steelblue', edgecolor='white')
ax.bar(x + width/2, theory,  width, label='理論値',           color='coral',     edgecolor='white')
ax.plot(x - width/2, probs,  'o-', color='steelblue', markersize=7)
ax.plot(x + width/2, theory, 's--', color='coral',    markersize=7)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('確率', fontsize=12)
ax.set_title('単調性：A1 $\subset$ A2 $\subset$ A3 $\subset$ A4  $\Rightarrow$  P(A1) $\leq$ P(A2) $\leq$ P(A3) $\leq$ P(A4)', fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 0.8)
plt.tight_layout()
plt.show()

print('事象の包含関係と確率の単調性：')
for (name, _), p, t in zip(nested, probs, theory):
    print(f'  P({name:12s}) = {p:.4f}  （理論値 {t:.4f}）')
print(f'\n単調増加を確認: {all(probs[i] <= probs[i+1] for i in range(len(probs)-1))}')

---
## 7. おまけ：大数の法則との接続

試行回数を増やすほど経験確率が真の確率に収束していく様子を観察します。  
これは確率の公理とシミュレーションの橋渡しとなる重要な定理です。

$$\frac{1}{n}\sum_{i=1}^n \mathbf{1}_{A}(\omega_i) \xrightarrow{n\to\infty} P(A)$$

In [ ]:
np.random.seed(0)
N = 50000
n_trials = np.logspace(1, np.log10(N), 300).astype(int)
n_trials = np.unique(n_trials)

# 3つの事象で収束を確認
rolls_large = np.random.randint(1, 7, size=N)
events_lln = [
    ('P(目=1)   = 1/6', rolls_large == 1, 1/6, '#4C72B0'),
    ('P(偶数)   = 1/2', rolls_large % 2 == 0, 1/2, '#DD8452'),
    ('P(4以上) = 1/2', rolls_large >= 4, 1/2, '#55A868'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, (label, mask, theory_p, color) in zip(axes, events_lln):
    freq_seq = [mask[:k].mean() for k in n_trials]
    ax.semilogx(n_trials, freq_seq, color=color, linewidth=1.2, alpha=0.8)
    ax.axhline(theory_p, color='crimson', linestyle='--', linewidth=1.5,
               label=f'理論値 = {theory_p:.4f}')
    ax.fill_between(n_trials,
                    theory_p - 0.02, theory_p + 0.02,
                    alpha=0.1, color='crimson')
    ax.set_xlabel('試行回数 n（対数スケール）', fontsize=10)
    ax.set_ylabel('経験確率', fontsize=10)
    ax.set_title(label, fontsize=11)
    ax.legend(fontsize=10)

fig.suptitle('大数の法則：経験確率の真の確率への収束', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('最終的な経験確率（n = 50,000）：')
for label, mask, theory_p, _ in events_lln:
    emp = mask.mean()
    print(f'  {label}  経験値={emp:.5f}  誤差={abs(emp-theory_p):.5f}')

---
## まとめ

| 性質 | 数式 | 確認 |
|------|------|------|
| 非負性（公理1） | $P(A) \geq 0$ | ✓ セル3 |
| 全確率（公理2） | $P(\Omega) = 1$ | ✓ セル4 |
| 加法性（公理3） | $P(A \cup B) = P(A)+P(B)$ （排反） | ✓ セル5 |
| 補事象 | $P(A^c) = 1 - P(A)$ | ✓ セル6-1 |
| 包除原理 | $P(A \cup B) = P(A)+P(B)-P(A \cap B)$ | ✓ セル6-2 |
| 単調性 | $A \subseteq B \Rightarrow P(A) \leq P(B)$ | ✓ セル6-3 |
| 大数の法則 | 経験確率 → 真の確率 | ✓ セル7 |

> **ポイント**：コルモゴロフの3公理は「最小限の仮定」であり、  
> これだけから確率論の全ての豊かな性質が論理的に導出されます。